In [ ]:
import copy
import json
import platform
import random
import time
import warnings
import itertools
from contextlib import contextmanager
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets, transforms, models
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    confusion_matrix, accuracy_score, precision_recall_fscore_support,
    f1_score, recall_score, roc_auc_score, brier_score_loss, roc_curve, auc
)
from sklearn.calibration import calibration_curve
from scipy import stats

warnings.filterwarnings("ignore", category=UserWarning)

# =============================================================================
# CONFIGURATION
# =============================================================================
SEED = 11
IMG_SIZE = 224
BATCH_SIZE = 32                      # fast GPU-friendly default
EPOCHS = 30                           # maximum training budget for every model/seed
EARLY_STOPPING_PATIENCE = 5           # stop after 5 validation epochs without material improvement
EARLY_STOPPING_MIN_DELTA = 1e-4
RUN_FIXED_30_EPOCHS = False           # max=30; restore the best validation checkpoint
FAST_MODE = True                     # freeze pretrained feature extractors; train classifier heads only
USE_AMP = torch.cuda.is_available()  # mixed precision on CUDA for faster training
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 0                      # safest on Windows
VAL_FRACTION = 0.15
TEST_FRACTION = 0.15
XAI_SAMPLES_PER_CLASS = 3             # fast but class-balanced qualitative/XAI sample
SMOOTHGRAD_SAMPLES = 2                # reduced for speed; protocol remains reproducible
SMOOTHGRAD_NOISE_STD = 0.10
ALIGNMENT_SALIENCY_QUANTILE = 0.75
REFERENCE_EDGE_QUANTILE = 0.70

# ---- Reproducible perturbation protocol (fully specified for reviewers) ---
NOISE_STD = 0.35
PERTURB_K = [1, 2, 5, 10, 20]
PERTURBATION_MAGNITUDE = {
    "noise": "Gaussian additive noise; sigma=0.35 applied to the normalized model input",
    "erase": "selected pixels replaced by 0 after the normalization pipeline",
}
PERTURBATION_AFFECTED_PIXEL_PERCENT = [1, 2, 5, 10, 20]
PERTURBATION_REPETITIONS = 5
PERTURBATION_RANDOM_SEEDS = [1101, 1102, 1103, 1104, 1105]
PERTURBATION_AGGREGATION = (
    "For each image and perturbation level: mean over 5 perturbation repetitions "
    "(seeds 1101-1105); then mean over held-out XAI-sample images; then one value per "
    "independent training run; final report = Mean, sample SD (ddof=1), Min and Max "
    "across the five independent training seeds 11, 22, 33, 44, 55."
)
TABLE5_UNITS_SCALE = (
    "Detection Strength and Stability are unitless scores in [0,1] internally and are "
    "reported multiplied by 100 (%) to match the manuscript's presentation."
)

# ---- Independent-seed reproducibility (used for every SD / min / max in this file) ----
REPEAT_SEEDS = [11, 22, 33, 44, 55]
FIVE_MODEL_PROTOCOL = "2D-CNN, VGG16, ResNet20, AlexNet, EfficientNet; identical split protocol and five independent seeds for both datasets."   #========================================================================================
N_REPEATS = len(REPEAT_SEEDS)
REUSE_PRIMARY_SEED_RUN = True   # seed==SEED reuses the already-trained primary run

CLASSIFICATION_METRICS = ["Accuracy", "Sensitivity", "Specificity", "F1", "AUC"]
CALIBRATION_BINS = 10

STATISTICAL_DVS = ["Precision", "Recall", "F1", "IoU", "Noise stability", "Erase stability"]
STAT_ALPHA = 0.05
STAT_BOOTSTRAP_REPS = 2000
STAT_BOOTSTRAP_SEED = 20260927
MULTIPLICITY_PROCEDURE = (
    "Holm step-down family-wise error control is applied separately to the six "
    "omnibus XAI-technique tests (Table 11) and the six planned DeepContour-vs-DeepEdge "
    "paired contrasts (Table 12)."
)


# ---- Ablation of the Unified Saliency-Edge framework -----------------------
FUSION_ALPHA, FUSION_BETA, FUSION_GAMMA = 0.50, 0.30, 0.20
POOL_REFINEMENT_STRENGTH = 0.20
KERNEL_REFINEMENT_STRENGTH = 0.20
KERNEL_WEIGHT_DELTA = 0.05
ABLATION_VARIANTS = [
    "Full",
    "w/o contour",
    "w/o edge",
    "w/o pooling",
    "w/o kernel manipulation",
    "w/o parameterized fusion",
    "Saliency only",
]

DATASET_PATHS = {
    "COVID-19": Path(r"C:\Users\HP\Desktop\DATASET\SARS_COV-2 CT_Scan Dataset"),             # Replace with your SARS_COV-2 CT Scan dataset path
    "Tuberculosis(TB)": Path(r"C:\Users\HP\Desktop\DATASET\TB_Chest_Radiography_Database"),  # Replace with your TB_Chest Radiography dataset path
}
PRIMARY_DATASET = "COVID-19"

SAVE_OUTPUTS = True
OUTPUT_DIR = Path("xai_outputs")
FIG_DIR = OUTPUT_DIR / "figures"
TABLE_DIR = OUTPUT_DIR / "tables"
ABLATION_DIR = OUTPUT_DIR / "ablation"
PRED_DIR = OUTPUT_DIR / "predictions"
CKPT_DIR = OUTPUT_DIR / "checkpoints"

MODEL_NAMES = ["2D-CNN", "VGG16", "ResNet20", "AlexNet", "EfficientNet"]
XAI_METHODS = ["GradCAM", "GradCAM++", "SmoothGrad", "LayerCAM", "DeepContour", "DeepEdge"]
TABLE3_XAI_ORDER = ["GradCAM++", "GradCAM", "SmoothGrad", "LayerCAM", "DeepContour", "DeepEdge"]

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def set_global_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True


set_global_seed(SEED)

# =============================================================================
# JOURNAL-STYLE PLOTTING THEME
# =============================================================================
OKABE_ITO = {
    "black": "#000000", "orange": "#E69F00", "sky": "#56B4E9", "green": "#009E73",
    "yellow": "#F0E442", "blue": "#0072B2", "vermillion": "#D55E00", "purple": "#CC79A7",
}
COLOR_MAX, COLOR_AVG = OKABE_ITO["blue"], OKABE_ITO["orange"]
COLOR_BAR = "#4C72B0"
COLOR_HIGHLIGHT = "#0B5AA6"
SINGLE_COL_IN, DOUBLE_COL_IN = 3.5, 7.2


def apply_journal_style():
    plt.rcParams.update({
        "figure.dpi": 300, "savefig.dpi": 600, "savefig.bbox": "tight",
        "font.family": "sans-serif", "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
        "font.size": 8, "axes.titlesize": 9, "axes.titleweight": "bold",
        "axes.labelsize": 8, "xtick.labelsize": 7, "ytick.labelsize": 7,
        "legend.fontsize": 7, "legend.frameon": False,
        "figure.titlesize": 12, "figure.titleweight": "bold",
        "axes.spines.top": False, "axes.spines.right": False,
        "axes.grid": True, "grid.alpha": 0.25, "grid.linewidth": 0.5,
        "lines.linewidth": 1.4, "lines.markersize": 4,
        "axes.prop_cycle": plt.cycler(color=list(OKABE_ITO.values())),
        "image.cmap": "viridis",
    })


apply_journal_style()
plt.ion()


def label_panel(ax, text):
    ax.text(-0.06, 1.10, text, transform=ax.transAxes, fontsize=9,
             fontweight="bold", va="top", ha="left")


def save_figure(fig, stem, dataset_name=None):
    if SAVE_OUTPUTS:
        FIG_DIR.mkdir(parents=True, exist_ok=True)
        safe_ds = (dataset_name or "").replace(" ", "_").replace("(", "").replace(")", "")
        name = f"{stem}_{safe_ds}" if safe_ds else stem
        fig.savefig(FIG_DIR / f"{name}.pdf")
        fig.savefig(FIG_DIR / f"{name}.png", dpi=600)
    plt.show()


print("Device:", DEVICE)
print("Python:", platform.python_version(), "| Torch:", torch.__version__)

# =============================================================================
# DATA
# =============================================================================
train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(0.5),
    transforms.RandomRotation(8),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
display_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
])


class TransformSubset(Dataset):
    def __init__(self, imagefolder, indices, transform):
        self.imagefolder = imagefolder
        self.indices = list(indices)
        self.transform = transform

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        real_idx = self.indices[i]
        path, label = self.imagefolder.samples[real_idx]
        img = Image.open(path).convert("RGB")
        return self.transform(img), label


class DatasetBundle:
    def __init__(self, name, path):
        self.name = name
        self.path = Path(path)
        base = datasets.ImageFolder(self.path)
        self.base_dataset = base
        self.class_names = base.classes
        self.num_classes = len(self.class_names)
        if self.num_classes < 2:
            raise ValueError(f"[{name}] needs at least two class folders for classification.")

        targets = np.asarray(base.targets)
        all_indices = np.arange(len(base))
        train_idx, temp_idx = train_test_split(
            all_indices, test_size=VAL_FRACTION + TEST_FRACTION,
            random_state=SEED, stratify=targets,
        )
        relative_test = TEST_FRACTION / (VAL_FRACTION + TEST_FRACTION)
        val_idx, test_idx = train_test_split(
            temp_idx, test_size=relative_test, random_state=SEED, stratify=targets[temp_idx],
        )
        self.train_idx, self.val_idx, self.test_idx = train_idx, val_idx, test_idx

        self.train_ds = TransformSubset(base, train_idx, train_tf)
        self.val_ds = TransformSubset(base, val_idx, eval_tf)
        self.test_ds = TransformSubset(base, test_idx, eval_tf)
        self.test_display_ds = TransformSubset(base, test_idx, display_tf)

        pin = torch.cuda.is_available()
        # Keep the original batch size/splits and Windows-safe worker setting.
        # persistent_workers is only valid when NUM_WORKERS > 0.
        loader_common = dict(
            batch_size=BATCH_SIZE,
            num_workers=NUM_WORKERS,
            pin_memory=pin,
            persistent_workers=(NUM_WORKERS > 0),
        )
        self.train_loader = DataLoader(self.train_ds, shuffle=True, **loader_common)
        self.val_loader = DataLoader(self.val_ds, shuffle=False, **loader_common)
        self.test_loader = DataLoader(self.test_ds, shuffle=False, **loader_common)

        print(f"[{name}] classes: {self.class_names} | train/val/test: "
              f"{len(train_idx)}/{len(val_idx)}/{len(test_idx)}")

        # Actual class-wise train/validation/test counts from the realized split.
        print(f"\n[{name}] CLASS-WISE DATASET SPLIT")
        print("-" * 60)
        for split_name, split_idx in (
            ("Train", train_idx),
            ("Validation", val_idx),
            ("Test", test_idx),
        ):
            split_targets = targets[split_idx]
            print(f"{split_name} (total={len(split_idx)}):")
            for class_id, class_name in enumerate(self.class_names):
                class_count = int(np.count_nonzero(split_targets == class_id))
                print(f"  {class_name}: {class_count}")
        print("-" * 60)

        # Also expose the exact realized split counts as a compact table.
        # This is generated independently for BOTH configured datasets because
        # every DatasetBundle executes this block when it is loaded.
        split_rows = []
        for split_name, split_idx in (
            ("Train", train_idx),
            ("Validation", val_idx),
            ("Test", test_idx),
        ):
            split_targets = targets[split_idx]
            row = {"Dataset": name, "Split": split_name, "Total": int(len(split_idx))}
            for class_id, class_name in enumerate(self.class_names):
                row[class_name] = int(np.count_nonzero(split_targets == class_id))
            split_rows.append(row)
        self.classwise_split_table = pd.DataFrame(split_rows)
        print(self.classwise_split_table.to_string(index=False))

        if SAVE_OUTPUTS:
            TABLE_DIR.mkdir(parents=True, exist_ok=True)
            safe_ds = name.replace(" ", "_").replace("(", "").replace(")", "")
            self.classwise_split_table.to_csv(
                TABLE_DIR / f"classwise_train_val_test_counts_{safe_ds}.csv",
                index=False,
            )

    def get_sample(self, local_idx):
        x, y = self.test_ds[local_idx]
        xd, _ = self.test_display_ds[local_idx]
        return x.unsqueeze(0), y, xd

    def choose_xai_indices(self):
        selected, counts = [], defaultdict(int)
        for local_i, global_i in enumerate(self.test_idx):
            y = self.base_dataset.targets[global_i]
            if counts[y] < XAI_SAMPLES_PER_CLASS:
                selected.append(local_i)
                counts[y] += 1
            if all(counts[c] >= XAI_SAMPLES_PER_CLASS for c in range(self.num_classes)):
                break
        return selected or list(range(min(8, len(self.test_ds))))


def load_available_datasets(paths):
    bundles = {}
    for name, path in paths.items():
        path = Path(path)
        if not path.exists():
            print(f"WARNING: dataset '{name}' not found at {path} -- skipping. "
                  f"Edit DATASET_PATHS to point at your data.")
            continue
        try:
            bundles[name] = DatasetBundle(name, path)
        except Exception as exc:
            print(f"WARNING: failed to load dataset '{name}' ({exc}) -- skipping.")
    if not bundles:
        raise FileNotFoundError(
            "No configured dataset paths exist on this machine. Edit DATASET_PATHS "
            "so at least one entry points at a folder containing one subfolder per class."
        )
    return bundles


# =============================================================================
# MODELS
# =============================================================================
class CNN2D(nn.Module):
    def __init__(self, n_classes):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(inplace=False), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(inplace=False), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(inplace=False), nn.MaxPool2d(2),
            nn.Conv2d(128, 256, 3, padding=1), nn.ReLU(inplace=False),
        )
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(nn.Flatten(), nn.Dropout(0.30), nn.Linear(256, n_classes))

    def forward(self, x):
        return self.classifier(self.pool(self.features(x)))



class ResNet20BasicBlock(nn.Module):
    """Basic residual block used by the requested 20-layer ResNet."""
    expansion = 1

    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(
            in_channels, out_channels, kernel_size=3, stride=stride,
            padding=1, bias=False
        )
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=False)
        self.conv2 = nn.Conv2d(
            out_channels, out_channels, kernel_size=3, stride=1,
            padding=1, bias=False
        )
        self.bn2 = nn.BatchNorm2d(out_channels)

        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1,
                          stride=stride, bias=False),
                nn.BatchNorm2d(out_channels),
            )
        else:
            self.shortcut = nn.Identity()

    def forward(self, x):
        identity = self.shortcut(x)
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out = self.relu(out + identity)
        return out


class ResNet20(nn.Module):
    """
    20-layer residual CNN: 1 stem convolution + 3 stages x 3 residual
    blocks x 2 convolutions + final classifier.  The stem uses stride=2
    so 224x224 radiographs/CT slices remain computationally practical.
    """
    def __init__(self, n_classes):
        super().__init__()
        self.in_channels = 16
        self.stem = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=False),
        )
        self.layer1 = self._make_layer(16, blocks=3, stride=1)
        self.layer2 = self._make_layer(32, blocks=3, stride=2)
        self.layer3 = self._make_layer(64, blocks=3, stride=2)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(64, n_classes)

    def _make_layer(self, out_channels, blocks, stride):
        layers = [ResNet20BasicBlock(self.in_channels, out_channels, stride)]
        self.in_channels = out_channels
        for _ in range(1, blocks):
            layers.append(ResNet20BasicBlock(self.in_channels, out_channels, 1))
        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.pool(x).flatten(1)
        return self.fc(x)


def safe_pretrained(builder, weights_enum):
    try:
        return builder(weights=weights_enum.DEFAULT)
    except Exception as e:
        print("Pretrained weights unavailable; falling back to random weights:", e)
        return builder(weights=None)


def disable_inplace_activations(module):
    for m in module.modules():
        if hasattr(m, "inplace"):
            try:
                m.inplace = False
            except Exception:
                pass


def build_model(name, n_classes):
    if name == "2D-CNN":
        model = CNN2D(n_classes)
    elif name == "VGG16":
        model = safe_pretrained(models.vgg16, models.VGG16_Weights)
        model.classifier[-1] = nn.Linear(model.classifier[-1].in_features, n_classes)
    elif name == "ResNet20":
        model = ResNet20(n_classes)
    elif name == "AlexNet":
        model = safe_pretrained(models.alexnet, models.AlexNet_Weights)
        model.classifier[-1] = nn.Linear(model.classifier[-1].in_features, n_classes)
    elif name == "EfficientNet":
        model = safe_pretrained(models.efficientnet_b0, models.EfficientNet_B0_Weights)
        model.classifier[-1] = nn.Linear(model.classifier[-1].in_features, n_classes)
    else:
        raise ValueError(name)
    disable_inplace_activations(model)
    if FAST_MODE and name not in ("2D-CNN", "ResNet20"):
        # Transfer-learning fast path: keep ImageNet feature extractor fixed and train only the head.
        for p in model.parameters():
            p.requires_grad = False
        if name in ("VGG16", "AlexNet"):
            for p in model.classifier[-1].parameters(): p.requires_grad = True
        elif name == "EfficientNet":
            for p in model.classifier[-1].parameters(): p.requires_grad = True
    return model.to(DEVICE)


def target_layer_for(name, model):
    if name == "2D-CNN":
        return model.features[9]
    if name == "VGG16":
        return [m for m in model.features.modules() if isinstance(m, nn.Conv2d)][-1]
    if name == "ResNet20":
        return model.layer3[-1].conv2
    if name == "AlexNet":
        return [m for m in model.features.modules() if isinstance(m, nn.Conv2d)][-1]
    if name == "EfficientNet":
        return [m for m in model.features.modules() if isinstance(m, nn.Conv2d)][-1]
    raise ValueError(name)


# TRAINING BUDGET AND EARLY STOPPING:
# All five architectures have a maximum budget of 30 epochs for each of the five
# independent seeds on each dataset. Training stops after 5 validation epochs
# without material validation-loss improvement; the best validation checkpoint
# is restored for held-out evaluation. Transfer-learning
# feature extractors remain frozen in FAST_MODE for VGG16/AlexNet/EfficientNet;
# 2D-CNN and ResNet20 are trained end-to-end.

# =============================================================================
# TRAINING
# =============================================================================
def run_epoch(model, loader, optimizer=None):
    training = optimizer is not None
    model.train(training)
    criterion = nn.CrossEntropyLoss()
    total_loss = 0.0
    y_true, y_pred = [], []
    for xb, yb in loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        if training:
            optimizer.zero_grad(set_to_none=True)
        with torch.set_grad_enabled(training):
            logits = model(xb)
            loss = criterion(logits, yb)
            if training:
                loss.backward()
                optimizer.step()
        total_loss += loss.item() * xb.size(0)
        y_true.extend(yb.detach().cpu().tolist())
        y_pred.extend(logits.argmax(1).detach().cpu().tolist())
    return total_loss / len(loader.dataset), accuracy_score(y_true, y_pred)


def train_one_model(model, name, bundle):
    trainable = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(trainable, lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=1)
    history = defaultdict(list)
    best_state = copy.deepcopy(model.state_dict())
    best_val_loss = float("inf")
    best_val_acc = -1.0
    best_epoch = 0
    stale_epochs = 0
    for epoch in range(1, EPOCHS + 1):
        t0 = time.time()
        tr_loss, tr_acc = run_epoch(model, bundle.train_loader, optimizer)
        va_loss, va_acc = run_epoch(model, bundle.val_loader, None)
        scheduler.step(va_acc)
        for k, v in [("train_loss", tr_loss), ("train_acc", tr_acc),
                     ("val_loss", va_loss), ("val_acc", va_acc)]:
            history[k].append(v)
        improved_loss = va_loss < (best_val_loss - EARLY_STOPPING_MIN_DELTA)
        tied_loss_better_acc = abs(va_loss - best_val_loss) <= EARLY_STOPPING_MIN_DELTA and va_acc > best_val_acc
        if improved_loss or tied_loss_better_acc:
            best_val_loss = va_loss
            best_val_acc = va_acc
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())
            stale_epochs = 0
        else:
            stale_epochs += 1
        print(f"[{bundle.name}] {name:12s} epoch {epoch:02d}/{EPOCHS} | "
              f"train {tr_acc:.4f} | val {va_acc:.4f} | {time.time()-t0:.1f}s")
        if (not RUN_FIXED_30_EPOCHS) and stale_epochs >= EARLY_STOPPING_PATIENCE:
            print(f"[{bundle.name}] {name}: early stopping at epoch {epoch}; best epoch={best_epoch}, best val loss={best_val_loss:.6f}, best val acc={best_val_acc:.4f}")
            break
    model.load_state_dict(best_state)
    return model, dict(history)


@torch.inference_mode()
def predict(model, loader):
    model.eval()
    ys, ps, probs = [], [], []
    for xb, yb in loader:
        logits = model(xb.to(DEVICE))
        prob = F.softmax(logits, dim=1)
        ys.extend(yb.numpy().tolist())
        ps.extend(prob.argmax(1).cpu().numpy().tolist())
        probs.extend(prob.cpu().numpy())
    return np.asarray(ys), np.asarray(ps), np.asarray(probs)


# =============================================================================
# IMAGE / METRIC UTILITIES
# =============================================================================
MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)


def denorm(x):
    return (x.detach().cpu() * STD + MEAN).clamp(0, 1)


def rgb_from_norm(x):
    return denorm(x).permute(1, 2, 0).numpy()


def rgb_from_display(x):
    return x.detach().cpu().permute(1, 2, 0).numpy().clip(0, 1)


def normalize_map(a):
    a = np.asarray(a, np.float32)
    a = a - np.nanmin(a)
    mx = np.nanmax(a)
    return np.nan_to_num(a / mx) if mx > 1e-12 else np.nan_to_num(a)


def gray(rgb):
    return 0.2989 * rgb[..., 0] + 0.5870 * rgb[..., 1] + 0.1140 * rgb[..., 2]


def edge_map(rgb):
    g = gray(rgb).astype(np.float32)
    gy, gx = np.gradient(g)
    return normalize_map(np.sqrt(gx * gx + gy * gy))


def proxy_reference_mask(rgb):
    e = edge_map(rgb)
    g = gray(rgb)
    h, w = g.shape
    yy, xx = np.mgrid[:h, :w]
    ellipse = ((xx - w / 2) / (0.48 * w)) ** 2 + ((yy - h / 2) / (0.48 * h)) ** 2 <= 1
    return ((e >= np.quantile(e, REFERENCE_EDGE_QUANTILE)) &
            (g > 0.05) & (g < 0.98) & ellipse).astype(np.uint8)


def binarize_saliency(s):
    s = normalize_map(s)
    return (s >= np.quantile(s, ALIGNMENT_SALIENCY_QUANTILE)).astype(np.uint8)


def binary_metrics(pred, ref):
    pred, ref = pred.astype(bool), ref.astype(bool)
    tp = np.logical_and(pred, ref).sum()
    fp = np.logical_and(pred, ~ref).sum()
    fn = np.logical_and(~pred, ref).sum()
    p = tp / (tp + fp + 1e-8)
    r = tp / (tp + fn + 1e-8)
    f1 = 2 * p * r / (p + r + 1e-8)
    iou = tp / (tp + fp + fn + 1e-8)
    return p, r, f1, iou


def binary_iou(a, b):
    a, b = a.astype(bool), b.astype(bool)
    return np.logical_and(a, b).sum() / (np.logical_or(a, b).sum() + 1e-8)


def center_bias_score(s):
    s = normalize_map(s)
    h, w = s.shape
    l, r = s[:, :w // 2].mean(), s[:, w // 2:].mean()
    t, b = s[:h // 2].mean(), s[h // 2:].mean()
    ah = abs(l - r) / (l + r + 1e-8)
    av = abs(t - b) / (t + b + 1e-8)
    return float(np.clip(1 - 0.5 * (ah + av), 0, 1))


# =============================================================================
# XAI ENGINE -- NO BACKWARD HOOKS (autograd.grad on real forward activations)
# =============================================================================
def grad_wrt(scalar_output, tensor, retain_graph=False, create_graph=False):
    grads = torch.autograd.grad(
        scalar_output, tensor, retain_graph=retain_graph,
        create_graph=create_graph, allow_unused=True,
    )
    grad = grads[0]
    if grad is None:
        warnings.warn(
            "Target tensor was disconnected from the autograd graph; returning zeros.",
            RuntimeWarning,
        )
        grad = torch.zeros_like(tensor)
    return grad


class ActivationCapture:
    def __init__(self, layer):
        self.activation = None
        self.handle = layer.register_forward_hook(self._hook)

    def _hook(self, module, inputs, output):
        self.activation = output

    def close(self):
        self.handle.remove()


def cam_core(model, model_name, x, target=None, method="GradCAM"):
    model.eval()
    cap = ActivationCapture(target_layer_for(model_name, model))
    try:
        x = x.clone().detach().to(DEVICE).requires_grad_(True)
        logits = model(x)
        target = int(logits.argmax(1)[0]) if target is None else int(target)
        score = logits[:, target].sum()
        act = cap.activation
        grad = grad_wrt(score, act, retain_graph=False, create_graph=False)
        act_np = act.detach().clone()
        if method == "GradCAM":
            weights = grad.mean((2, 3), keepdim=True)
            cam = F.relu((weights * act_np).sum(1))
        elif method == "GradCAM++":
            g2, g3 = grad.pow(2), grad.pow(3)
            alpha = g2 / (2 * g2 + act_np.sum((2, 3), keepdim=True) * g3 + 1e-8)
            weights = (alpha * F.relu(grad)).sum((2, 3), keepdim=True)
            cam = F.relu((weights * act_np).sum(1))
        elif method == "LayerCAM":
            cam = F.relu((F.relu(grad) * act_np).sum(1))
        else:
            raise ValueError(method)
        cam = F.interpolate(cam[:, None], (IMG_SIZE, IMG_SIZE), mode="bilinear",
                             align_corners=False)[:, 0]
        return normalize_map(cam[0].detach().cpu().numpy())
    finally:
        cap.close()


def smoothgrad(model, x, target=None):
    model.eval()
    x0 = x.clone().detach().to(DEVICE)
    with torch.no_grad():
        target = int(model(x0).argmax(1)[0]) if target is None else int(target)
    acc = torch.zeros_like(x0)
    for _ in range(SMOOTHGRAD_SAMPLES):
        xn = (x0 + SMOOTHGRAD_NOISE_STD * torch.randn_like(x0)).requires_grad_(True)
        score = model(xn)[:, target].sum()
        g = grad_wrt(score, xn, retain_graph=False, create_graph=False)
        acc += g.abs().detach()
    return normalize_map(acc.mean(0).mean(0).cpu().numpy())


def deep_edge_like(model, model_name, x, target=None):
    base = cam_core(model, model_name, x, target, "GradCAM")
    return normalize_map(0.60 * base + 0.40 * edge_map(rgb_from_norm(x[0])))


def deep_contour_like(model, model_name, x, target=None):
    base = cam_core(model, model_name, x, target, "GradCAM")
    rgb = rgb_from_norm(x[0])
    g = gray(rgb).astype(np.float32)
    gy, gx = np.gradient(g)
    contour = normalize_map(np.abs(gx) + np.abs(gy))
    return normalize_map(0.50 * base + 0.30 * edge_map(rgb) + 0.20 * contour)


def saliency(model, model_name, x, method, target=None):
    if method == "GradCAM":
        return cam_core(model, model_name, x, target, "GradCAM")
    if method == "GradCAM++":
        return cam_core(model, model_name, x, target, "GradCAM++")
    if method == "SmoothGrad":
        return smoothgrad(model, x, target)
    if method == "LayerCAM":
        return cam_core(model, model_name, x, target, "LayerCAM")
    if method == "DeepContour":
        return deep_contour_like(model, model_name, x, target)
    if method == "DeepEdge":
        return deep_edge_like(model, model_name, x, target)
    raise ValueError(method)


# =============================================================================
# PERTURBATION AND METRICS
# =============================================================================
def perturb_by_saliency(x, s, k_percent, mode, rng_seed=None):
    x2 = x.clone().detach()
    if rng_seed is not None:
        gen = torch.Generator(device="cpu").manual_seed(rng_seed)
    else:
        gen = None
    st = torch.tensor(s, device=x2.device, dtype=torch.float32)
    flat = st.flatten()
    n = max(1, int(round(k_percent / 100 * flat.numel())))
    inds = torch.topk(flat, n).indices
    mask = torch.zeros_like(flat, dtype=torch.bool)
    mask[inds] = True
    mask = mask.view(st.shape)[None, None].expand_as(x2)
    if mode == "erase":
        x2[mask] = 0.0
    elif mode == "noise":
        noise = torch.randn(x2.shape, generator=gen).to(x2.device) if gen is not None else torch.randn_like(x2)
        x2 = torch.where(mask, x2 + NOISE_STD * noise, x2)
    else:
        raise ValueError(mode)
    return x2


@torch.inference_mode()
def probvec(model, x):
    return F.softmax(model(x.to(DEVICE)), 1)[0].cpu().numpy()


def stability_over_repetitions(model, model_name, x, method, mode, k_percent):
    """Reproducible stability under a fixed perturbation protocol: mean IoU
    of the binarized saliency map before/after perturbation, averaged over
    PERTURBATION_REPETITIONS repeats using PERTURBATION_RANDOM_SEEDS."""
    with torch.no_grad():
        target = int(model(x.to(DEVICE)).argmax(1)[0])
    s0 = saliency(model, model_name, x, method, target)
    m0 = binarize_saliency(s0)
    ious = []
    for rep in range(PERTURBATION_REPETITIONS):
        rng_seed = PERTURBATION_RANDOM_SEEDS[rep % len(PERTURBATION_RANDOM_SEEDS)]
        xp = perturb_by_saliency(x.to(DEVICE), s0, k_percent, mode, rng_seed=rng_seed)
        s1 = saliency(model, model_name, xp, method, target)
        ious.append(binary_iou(m0, binarize_saliency(s1)))
    return float(np.mean(ious))


def detection_strength(s):
    a = normalize_map(s).ravel()
    n = max(1, int(0.10 * len(a)))
    return float(100 * np.partition(a, -n)[-n:].mean())


def evaluate_xai(model, model_name, method, bundle, xai_indices):
    align, ns, es, interps, biases, strengths = [], [], [], [], [], []
    for local_idx in xai_indices:
        x, y, xd = bundle.get_sample(local_idx)
        x = x.to(DEVICE)
        with torch.no_grad():
            pred = int(model(x).argmax(1)[0])
        s = saliency(model, model_name, x, method, pred)
        ref = proxy_reference_mask(rgb_from_display(xd))
        sm = binarize_saliency(s)
        p, r, f1, iou = binary_metrics(sm, ref)
        align.append((p, r, f1, iou))
        ns.append(stability_over_repetitions(model, model_name, x, method, "noise", 10))
        es.append(stability_over_repetitions(model, model_name, x, method, "erase", 10))
        interps.append(0.5 * (f1 + iou))
        biases.append(center_bias_score(s))
        strengths.append(detection_strength(s))
    a = np.asarray(align)
    return {
        "Precision": a[:, 0].mean(), "Recall": a[:, 1].mean(), "F1": a[:, 2].mean(), "IoU": a[:, 3].mean(),
        "Noise stability": np.mean(ns), "Erase stability": np.mean(es),
        "Interpretability": np.mean(interps), "Bias Score": np.mean(biases),
        "Detection Strength": np.mean(strengths),
        "Stability": 100 * 0.5 * (np.mean(ns) + np.mean(es)),
    }


def capture_feature_map(model, layer, x):
    holder = {}

    def hook(m, i, o):
        holder["feat"] = o.detach().clone()

    h = layer.register_forward_hook(hook)
    try:
        with torch.no_grad():
            model(x)
        return holder["feat"][0].cpu()
    finally:
        h.remove()


def sensitivity_curve(model, model_name, method, x, mode):
    with torch.no_grad():
        target = int(model(x).argmax(1)[0])
    s = saliency(model, model_name, x, method, target)
    p0 = probvec(model, x)
    maxc, avgc = [], []
    for k in PERTURB_K:
        p1 = probvec(model, perturb_by_saliency(x, s, k, mode))
        d = np.abs(p1 - p0)
        maxc.append(d.max())
        avgc.append(d.mean())
    return np.asarray(maxc), np.asarray(avgc)


def classification_metric_dict(y_true, y_pred, probs):
    y_true, y_pred, probs = np.asarray(y_true), np.asarray(y_pred), np.asarray(probs)
    n_classes = probs.shape[1]
    cm = confusion_matrix(y_true, y_pred, labels=np.arange(n_classes))
    total = cm.sum()
    spec_vals = []
    for c in range(n_classes):
        tp = cm[c, c]
        fn = cm[c, :].sum() - tp
        fp = cm[:, c].sum() - tp
        tn = total - tp - fn - fp
        spec_vals.append(tn / (tn + fp) if (tn + fp) else np.nan)
    out = {
        "Accuracy": float(accuracy_score(y_true, y_pred)),
        "Sensitivity": float(recall_score(y_true, y_pred, average="macro", zero_division=0)),
        "Specificity": float(np.nanmean(spec_vals)),
        "F1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
    }
    try:
        if n_classes == 2:
            out["AUC"] = float(roc_auc_score(y_true, probs[:, 1]))
            out["Brier"] = float(brier_score_loss(y_true, probs[:, 1]))
        else:
            out["AUC"] = float(roc_auc_score(y_true, probs, multi_class="ovr", average="macro"))
            y_onehot = np.eye(n_classes)[y_true]
            out["Brier"] = float(np.mean(np.sum((probs - y_onehot) ** 2, axis=1)))
    except ValueError:
        out["AUC"], out["Brier"] = np.nan, np.nan
    conf = probs.max(axis=1)
    corr = (y_pred == y_true).astype(float)
    out["ECE"] = expected_calibration_error(conf, corr, CALIBRATION_BINS)
    return out


def expected_calibration_error(confidence, correctness, n_bins=10):
    confidence = np.asarray(confidence, float)
    correctness = np.asarray(correctness, float)
    edges = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        mask = (confidence > lo) & (confidence <= hi)
        if mask.any():
            ece += mask.mean() * abs(correctness[mask].mean() - confidence[mask].mean())
    return float(ece)


# =============================================================================
# CROSS-SEED AGGREGATION HELPERS  (Mean, sample SD ddof=1, Min, Max)
# =============================================================================
def mean_sd_min_max(values):
    a = np.asarray([v for v in values if v is not None and np.isfinite(v)], dtype=float)
    if len(a) == 0:
        return dict(Mean=np.nan, SD=np.nan, Min=np.nan, Max=np.nan, N=0)
    return dict(
        Mean=float(np.mean(a)),
        SD=float(np.std(a, ddof=1)) if len(a) > 1 else 0.0,
        Min=float(np.min(a)), Max=float(np.max(a)), N=int(len(a)),
    )


def fmt_stat(d, decimals=3, scale=1.0):
    if d["N"] == 0 or not np.isfinite(d["Mean"]):
        return "n/a"
    m, sd, lo, hi = d["Mean"] * scale, d["SD"] * scale, d["Min"] * scale, d["Max"] * scale
    return f"{m:.{decimals}f} \u00b1 {sd:.{decimals}f} [{lo:.{decimals}f}, {hi:.{decimals}f}]"



def _render_table_image(df, title, filename, figsize, note=None, header_note=None):
    fig, ax = plt.subplots(figsize=figsize)
    ax.axis("off")
    full_title = title if not header_note else f"{title}\n{header_note}"
    ax.set_title(full_title, fontweight="bold", pad=14, fontsize=9)
    tab = ax.table(cellText=df.values, colLabels=df.columns, loc="center", cellLoc="center")
    tab.auto_set_font_size(False)
    tab.set_fontsize(7)
    tab.auto_set_column_width(col=list(range(len(df.columns))))
    tab.scale(1.05, 1.55)
    for (row_i, col_i), cell in tab.get_celld().items():
        cell.set_edgecolor("#B0B0B0")
        if row_i == 0:
            cell.set_text_props(fontweight="bold", color="white")
            cell.set_facecolor("#2F5C8A")
    if note:
        ax.text(0.0, -0.06, note, transform=ax.transAxes, fontsize=6.3, style="italic")
    plt.tight_layout()
    if SAVE_OUTPUTS:
        TABLE_DIR.mkdir(parents=True, exist_ok=True)
        fig.savefig(TABLE_DIR / f"{filename}.pdf")
        fig.savefig(TABLE_DIR / f"{filename}.png", dpi=600)
        df.to_csv(TABLE_DIR / f"{filename}.csv", index=False)
        df.to_json(TABLE_DIR / f"{filename}.json", orient="records", indent=2)
    plt.show()


# =============================================================================
# PER-SEED PIPELINE
# =============================================================================
def run_seed_pipeline(bundle, seed, model_names, cache=None):
    """Trains `model_names` for one independent seed and computes every
    XAI-technique metric for each of them. Returns per-seed raw results.
    `cache` (optional dict keyed by (dataset,seed)) lets later stages
    (ablation) reuse already-trained models instead of retraining."""
    set_global_seed(seed)
    trained_models, test_predictions, all_xai_metrics = {}, {}, {}
    for model_name in model_names:
        print(f"\n[{bundle.name} | seed {seed}] Training {model_name}")
        model = build_model(model_name, bundle.num_classes)
        model, _ = train_one_model(model, model_name, bundle)
        trained_models[model_name] = model
        test_predictions[model_name] = predict(model, bundle.test_loader)
        yt, yp, _ = test_predictions[model_name]
        print(f"[{bundle.name} | seed {seed}] {model_name} test accuracy: {accuracy_score(yt, yp):.4f}")

    xai_indices = bundle.choose_xai_indices()
    for model_name, model in trained_models.items():
        all_xai_metrics[model_name] = {}
        for method in XAI_METHODS:
            all_xai_metrics[model_name][method] = evaluate_xai(model, model_name, method, bundle, xai_indices)

    if SAVE_OUTPUTS:
        CKPT_DIR.mkdir(parents=True, exist_ok=True)
        PRED_DIR.mkdir(parents=True, exist_ok=True)
        safe_ds = bundle.name.replace(" ", "_").replace("(", "").replace(")", "")
        for model_name, model in trained_models.items():
            torch.save(model.state_dict(), CKPT_DIR / f"{safe_ds}__seed_{seed}__{model_name}.pt")
            yt, yp, pr = test_predictions[model_name]
            d = pd.DataFrame({"y_true": yt, "y_pred": yp})
            for c in range(pr.shape[1]):
                d[f"prob_class_{c}"] = pr[:, c]
            d.to_csv(PRED_DIR / f"{safe_ds}__seed_{seed}__{model_name}.csv", index=False)

    result = dict(seed=seed, trained_models=trained_models, test_predictions=test_predictions,
                  all_xai_metrics=all_xai_metrics, xai_indices=xai_indices)
    if cache is not None:
        cache[(bundle.name, seed)] = result
    return result


def run_all_seeds(bundle, model_names=MODEL_NAMES, cache=None):
    per_seed = {}
    for seed in REPEAT_SEEDS:
        if REUSE_PRIMARY_SEED_RUN and seed == SEED and (bundle.name, seed) in (cache or {}):
            per_seed[seed] = cache[(bundle.name, seed)]
            continue
        per_seed[seed] = run_seed_pipeline(bundle, seed, model_names, cache=cache)
    return per_seed


# =============================================================================
# TABLE 1 / 2 / 3 -- cross-seed Mean +/- SD [Min, Max]
# =============================================================================
def pick_best_model_name(primary_seed_result):
    return max(MODEL_NAMES, key=lambda n: accuracy_score(*primary_seed_result["test_predictions"][n][:2]))


def build_technique_seed_metric(per_seed, best_model_name):
    """Returns technique_seed_metric[metric][technique] = [values across seeds],
    using the SAME architecture (best_model_name) in every seed so techniques
    are compared on a fixed backbone."""
    metrics_of_interest = ["Precision", "Recall", "F1", "IoU", "Noise stability",
                            "Erase stability", "Interpretability", "Bias Score"]
    out = {m: {t: [] for t in XAI_METHODS} for m in metrics_of_interest}
    for seed, res in per_seed.items():
        for t in XAI_METHODS:
            md = res["all_xai_metrics"][best_model_name][t]
            for m in metrics_of_interest:
                out[m][t].append(md[m])
    return out


def build_table1(dataset_name, technique_seed_metric):
    rows = []
    for t in XAI_METHODS:
        rows.append({
            "Dataset": dataset_name, "Technique": t,
            "Precision": fmt_stat(mean_sd_min_max(technique_seed_metric["Precision"][t])),
            "Recall": fmt_stat(mean_sd_min_max(technique_seed_metric["Recall"][t])),
            "F1": fmt_stat(mean_sd_min_max(technique_seed_metric["F1"][t])),
            "IoU": fmt_stat(mean_sd_min_max(technique_seed_metric["IoU"][t])),
        })
    return pd.DataFrame(rows)


def build_table2(dataset_name, technique_seed_metric):
    rows = []
    for t in XAI_METHODS:
        rows.append({
            "Dataset": dataset_name, "Technique": t,
            "Noise stability": fmt_stat(mean_sd_min_max(technique_seed_metric["Noise stability"][t])),
            "Erase stability": fmt_stat(mean_sd_min_max(technique_seed_metric["Erase stability"][t])),
            "Interpretability": fmt_stat(mean_sd_min_max(technique_seed_metric["Interpretability"][t])),
            "Bias Score": fmt_stat(mean_sd_min_max(technique_seed_metric["Bias Score"][t])),
        })
    return pd.DataFrame(rows)


def build_table3(dataset_name, per_seed):
    """
    Table 3 -- uploaded manuscript layout:
      rows: GradCAM++, GradCAM, SmoothGrad, LayerCAM, DeepContour, DeepEdge
      columns: Detection Strength and Stability under each of
               2D-CNN, VGG16, ResNet20, AlexNet, EfficientNet.

    Every metric is reported as Mean ± SD [Min, Max] across independent
    training seeds 11, 22, 33, 44 and 55. Values are percentages.
    """
    ds_lists = {m: {t: [] for t in XAI_METHODS} for m in MODEL_NAMES}
    st_lists = {m: {t: [] for t in XAI_METHODS} for m in MODEL_NAMES}

    for seed in REPEAT_SEEDS:
        res = per_seed[seed]
        for model_name in MODEL_NAMES:
            for technique in XAI_METHODS:
                metrics = res["all_xai_metrics"][model_name][technique]
                ds_lists[model_name][technique].append(metrics["Detection Strength"])
                st_lists[model_name][technique].append(metrics["Stability"])

    rows = []
    for technique in TABLE3_XAI_ORDER:
        row = {"XAI Metrics": technique}
        for model_name in MODEL_NAMES:
            row[f"{model_name}\nDetection Strength"] = fmt_stat(
                mean_sd_min_max(ds_lists[model_name][technique]), decimals=2
            )
            row[f"{model_name}\nStability"] = fmt_stat(
                mean_sd_min_max(st_lists[model_name][technique]), decimals=2
            )
        rows.append(row)

    return pd.DataFrame(rows)


def render_table3(dataset_name, per_seed):
    """Render/export Table 3 separately for each dataset."""
    table3 = build_table3(dataset_name, per_seed)
    _render_table_image(
        table3,
        f"Table 3: Localization accuracy and stability metrics across models and XAI techniques — {dataset_name}",
        f"table3_localization_accuracy_stability_{dataset_name}",
        figsize=(DOUBLE_COL_IN * 2.55, 1.8 + 0.46 * len(table3)),
        note=(
            f"Every cell is Mean ± SD [Min, Max] across independent training seeds "
            f"{REPEAT_SEEDS}. Detection Strength and Stability are reported in percent."
        ),
    )
    return table3



# =============================================================================
# TABLES 11-12 -- repeated-measures statistical analysis across XAI techniques
# =============================================================================
def holm_adjust(p_values):
    """Holm step-down adjusted p-values, preserving input order."""
    p = np.asarray(p_values, dtype=float)
    m = len(p)
    order = np.argsort(p)
    adjusted_sorted = np.empty(m, dtype=float)
    running = 0.0
    for rank, idx in enumerate(order):
        candidate = (m - rank) * p[idx]
        running = max(running, candidate)
        adjusted_sorted[rank] = min(1.0, running)
    out = np.empty(m, dtype=float)
    for rank, idx in enumerate(order):
        out[idx] = adjusted_sorted[rank]
    return out


def rm_anova_oneway(matrix):
    """
    One-factor repeated-measures ANOVA.
    matrix shape = subjects/runs x conditions/XAI techniques.
    With 5 independent training seeds and 6 techniques: df=(5,20).
    """
    x = np.asarray(matrix, dtype=float)
    n, k = x.shape
    grand = x.mean()
    subj_means = x.mean(axis=1)
    cond_means = x.mean(axis=0)

    ss_total = np.sum((x - grand) ** 2)
    ss_subject = k * np.sum((subj_means - grand) ** 2)
    ss_condition = n * np.sum((cond_means - grand) ** 2)
    ss_error = ss_total - ss_subject - ss_condition

    df_condition = k - 1
    df_error = (n - 1) * (k - 1)
    ms_condition = ss_condition / df_condition
    ms_error = ss_error / df_error
    F = ms_condition / ms_error if ms_error > 0 else np.inf
    p = stats.f.sf(F, df_condition, df_error)
    partial_eta2 = ss_condition / (ss_condition + ss_error) if (ss_condition + ss_error) > 0 else np.nan
    return {
        "F": float(F), "df1": int(df_condition), "df2": int(df_error),
        "p": float(p), "partial_eta2": float(partial_eta2),
        "ss_condition": float(ss_condition), "ss_error": float(ss_error),
    }


def mauchly_sphericity(matrix):
    """
    Mauchly's test using an orthonormal contrast-space covariance matrix.
    Returns W, chi-square approximation, df and p.
    """
    x = np.asarray(matrix, dtype=float)
    n, k = x.shape
    # Helmert-like orthonormal basis spanning the k-1 dimensional contrast space.
    C = np.zeros((k - 1, k), dtype=float)
    for i in range(1, k):
        C[i - 1, :i] = 1.0 / np.sqrt(i * (i + 1))
        C[i - 1, i] = -i / np.sqrt(i * (i + 1))
    z = x @ C.T
    S = np.cov(z, rowvar=False, ddof=1)
    eig = np.linalg.eigvalsh(S)
    eig = np.clip(eig, 1e-12, None)
    p_dim = k - 1
    W = np.prod(eig) / ((eig.mean()) ** p_dim)
    W = float(np.clip(W, 1e-300, 1.0))
    # Common finite-sample correction for the chi-square approximation.
    correction = (2 * p_dim**2 + p_dim + 2) / (6 * p_dim * (n - 1))
    chi2 = -(n - 1) * (1 - correction) * np.log(W)
    df = p_dim * (p_dim + 1) / 2 - 1
    pval = stats.chi2.sf(chi2, df)
    return float(W), float(chi2), int(df), float(pval)


def greenhouse_geisser_epsilon(matrix):
    """Greenhouse-Geisser epsilon from the centered condition covariance."""
    x = np.asarray(matrix, dtype=float)
    k = x.shape[1]
    S = np.cov(x, rowvar=False, ddof=1)
    H = np.eye(k) - np.ones((k, k)) / k
    A = H @ S @ H
    tr = np.trace(A)
    denom = (k - 1) * np.trace(A @ A)
    if denom <= 0:
        return 1.0
    eps = (tr * tr) / denom
    return float(np.clip(eps, 1.0 / (k - 1), 1.0))


def shapiro_rm_residuals(matrix):
    """Shapiro-Wilk diagnostic on repeated-measures residuals."""
    x = np.asarray(matrix, dtype=float)
    residuals = x - x.mean(axis=1, keepdims=True) - x.mean(axis=0, keepdims=True) + x.mean()
    vals = residuals.ravel()
    if len(vals) < 3:
        return np.nan, np.nan
    W, p = stats.shapiro(vals)
    return float(W), float(p)


def partial_eta2_bootstrap_ci(matrix, reps=STAT_BOOTSTRAP_REPS, seed=STAT_BOOTSTRAP_SEED):
    """Seed-level nonparametric bootstrap CI for partial eta-squared."""
    x = np.asarray(matrix, dtype=float)
    n = x.shape[0]
    rng = np.random.default_rng(seed)
    vals = []
    for _ in range(reps):
        idx = rng.integers(0, n, size=n)
        try:
            v = rm_anova_oneway(x[idx])["partial_eta2"]
            if np.isfinite(v):
                vals.append(v)
        except Exception:
            pass
    if not vals:
        return np.nan, np.nan
    return tuple(np.quantile(vals, [0.025, 0.975]))


def paired_effect_ci(a, b, alpha=STAT_ALPHA):
    """DeepContour minus DeepEdge: paired t, mean-difference CI, Cohen's dz."""
    a, b = np.asarray(a, float), np.asarray(b, float)
    d = a - b
    n = len(d)
    mean_d = float(np.mean(d))
    sd_d = float(np.std(d, ddof=1))
    se = sd_d / np.sqrt(n) if n > 1 else np.nan
    tval = mean_d / se if se and se > 0 else np.nan
    df = n - 1
    p = 2 * stats.t.sf(abs(tval), df) if np.isfinite(tval) else np.nan
    crit = stats.t.ppf(1 - alpha / 2, df)
    ci_low, ci_high = mean_d - crit * se, mean_d + crit * se
    dz = mean_d / sd_d if sd_d > 0 else np.nan
    return mean_d, float(tval), int(df), float(p), float(ci_low), float(ci_high), float(dz)


def build_tables11_12(dataset_name, technique_seed_metric):
    """
    Table 11: one repeated-measures ANOVA per dependent variable.
    Table 12: one planned paired t-test per dependent variable
              (DeepContour - DeepEdge).
    Independent variable: XAI technique (6 within-run levels).
    Repeated experimental unit: independent training seed (n=5).
    """
    omnibus_rows, contrast_rows = [], []
    raw_omnibus_p, raw_contrast_p = [], []

    for j, metric in enumerate(STATISTICAL_DVS):
        matrix = np.array(
            [[technique_seed_metric[metric][tech][i] for tech in XAI_METHODS]
             for i in range(len(REPEAT_SEEDS))],
            dtype=float,
        )
        an = rm_anova_oneway(matrix)
        sh_w, sh_p = shapiro_rm_residuals(matrix)
        mw, mchi, mdf, mp = mauchly_sphericity(matrix)
        eps = greenhouse_geisser_epsilon(matrix)

        # If sphericity is rejected, use GG-adjusted degrees of freedom/p-value.
        if np.isfinite(mp) and mp < STAT_ALPHA:
            report_df1 = eps * an["df1"]
            report_df2 = eps * an["df2"]
            report_p = float(stats.f.sf(an["F"], report_df1, report_df2))
            correction = "Greenhouse-Geisser"
        else:
            report_df1, report_df2, report_p = an["df1"], an["df2"], an["p"]
            correction = "Uncorrected"

        eta_lo, eta_hi = partial_eta2_bootstrap_ci(
            matrix, seed=STAT_BOOTSTRAP_SEED + j
        )
        raw_omnibus_p.append(report_p)
        omnibus_rows.append({
            "Dependent variable": metric,
            "F": an["F"],
            "df1": report_df1,
            "df2": report_df2,
            "p_raw": report_p,
            "Partial eta^2": an["partial_eta2"],
            "95% CI eta^2": f"[{eta_lo:.3f}, {eta_hi:.3f}]",
            "Shapiro W": sh_w,
            "Shapiro p": sh_p,
            "Mauchly W": mw,
            "Mauchly p": mp,
            "GG epsilon": eps,
            "Sphericity handling": correction,
        })

        a = technique_seed_metric[metric]["DeepContour"]
        b = technique_seed_metric[metric]["DeepEdge"]
        md, tv, df, pv, lo, hi, dz = paired_effect_ci(a, b)
        raw_contrast_p.append(pv)
        contrast_rows.append({
            "Dependent variable": metric,
            "Mean difference": md,
            "t": tv,
            "df": df,
            "p_raw": pv,
            "95% CI": f"[{lo:.6f}, {hi:.6f}]",
            "Cohen dz": dz,
        })

    omni_adj = holm_adjust(raw_omnibus_p)
    cont_adj = holm_adjust(raw_contrast_p)
    for row, padj in zip(omnibus_rows, omni_adj):
        row["p_Holm"] = float(padj)
    for row, padj in zip(contrast_rows, cont_adj):
        row["p_Holm"] = float(padj)

    t11 = pd.DataFrame(omnibus_rows)
    t12 = pd.DataFrame(contrast_rows)

    # Publication-facing formatted copies; raw CSVs retain full numeric precision.
    t11_fmt = t11.copy()
    for c in ["F", "df1", "df2", "p_raw", "p_Holm", "Partial eta^2",
              "Shapiro W", "Shapiro p", "Mauchly W", "Mauchly p", "GG epsilon"]:
        t11_fmt[c] = t11_fmt[c].map(lambda v: f"{v:.6f}" if np.isfinite(v) else "n/a")
    t12_fmt = t12.copy()
    for c in ["Mean difference", "t", "p_raw", "p_Holm", "Cohen dz"]:
        t12_fmt[c] = t12_fmt[c].map(lambda v: f"{v:.6f}" if np.isfinite(v) else "n/a")

    return t11, t12, t11_fmt, t12_fmt


def render_tables11_12(dataset_name, technique_seed_metric):
    t11, t12, t11_fmt, t12_fmt = build_tables11_12(dataset_name, technique_seed_metric)

    _render_table_image(
        t11_fmt,
        f"Table 11: Repeated-measures XAI-technique analysis — {dataset_name}",
        f"table11_rm_anova_{dataset_name}",
        figsize=(DOUBLE_COL_IN * 2.8, 2.3 + 0.48 * len(t11_fmt)),
        note=(
            "One repeated-measures ANOVA per dependent variable; XAI technique is the "
            "within-run factor (6 levels), seed is the repeated experimental unit (n=5). "
            "Shapiro-Wilk residual and Mauchly sphericity diagnostics are shown. "
            "Greenhouse-Geisser correction is used when Mauchly p<0.05. "
            "Holm correction controls the six-test omnibus family. Partial eta^2 is "
            f"reported with a {STAT_BOOTSTRAP_REPS}-replicate seed-level bootstrap 95% CI."
        ),
    )
    _render_table_image(
        t12_fmt,
        f"Table 12: Planned DeepContour–DeepEdge paired contrasts — {dataset_name}",
        f"table12_deepcontour_deepedge_{dataset_name}",
        figsize=(DOUBLE_COL_IN * 1.8, 2.1 + 0.46 * len(t12_fmt)),
        note=(
            "Mean difference = DeepContour - DeepEdge. One paired t-test per dependent "
            "variable, df=4 for five paired seed-level observations. 95% CIs and Cohen's dz "
            "are reported; Holm correction controls the six planned contrasts."
        ),
    )

    if SAVE_OUTPUTS:
        TABLE_DIR.mkdir(parents=True, exist_ok=True)
        safe_ds = dataset_name.replace(" ", "_").replace("(", "").replace(")", "")
        t11.to_csv(TABLE_DIR / f"table11_rm_anova_raw_{safe_ds}.csv", index=False)
        t12.to_csv(TABLE_DIR / f"table12_planned_contrasts_raw_{safe_ds}.csv", index=False)

    return t11_fmt, t12_fmt


# =============================================================================
# CLASSIFICATION METRICS, CALIBRATION, CONFUSION MATRICES (cross-seed + primary)
# =============================================================================
def build_classification_table(dataset_name, per_seed):
    rows = []
    for model_name in MODEL_NAMES:
        per_metric = {k: [] for k in CLASSIFICATION_METRICS + ["ECE", "Brier"]}
        for seed, res in per_seed.items():
            yt, yp, pr = res["test_predictions"][model_name]
            d = classification_metric_dict(yt, yp, pr)
            for k in per_metric:
                per_metric[k].append(d[k])
        row = {"Dataset": dataset_name, "Model": model_name}
        for k in per_metric:
            row[k] = fmt_stat(mean_sd_min_max(per_metric[k]))
        rows.append(row)
    return pd.DataFrame(rows)


def draw_reliability_diagrams(dataset_name, primary_result):
    fig, ax = plt.subplots(figsize=(SINGLE_COL_IN * 1.35, SINGLE_COL_IN * 1.05))
    ax.plot([0, 1], [0, 1], "--", linewidth=1, label="Perfect calibration")
    for model_name in MODEL_NAMES:
        yt, yp, pr = primary_result["test_predictions"][model_name]
        conf = pr.max(1); corr = (yp == yt).astype(int)
        frac_pos, mean_pred = calibration_curve(corr, conf, n_bins=CALIBRATION_BINS, strategy="uniform")
        m = classification_metric_dict(yt, yp, pr)
        ax.plot(mean_pred, frac_pos, marker="o", label=f"{model_name} (ECE={m['ECE']:.3f})")
    ax.set_xlabel("Mean predicted confidence"); ax.set_ylabel("Empirical accuracy")
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.set_title(f"Reliability diagram (seed {SEED}) \u2014 {dataset_name}")
    ax.legend(fontsize=6)
    save_figure(fig, "calibration_reliability_all_models", dataset_name)


def draw_roc_curves(dataset_name, primary_result, n_classes):
    fig, ax = plt.subplots(figsize=(SINGLE_COL_IN * 1.35, SINGLE_COL_IN * 1.05))
    for model_name in MODEL_NAMES:
        yt, yp, pr = primary_result["test_predictions"][model_name]
        if n_classes == 2:
            fpr, tpr, _ = roc_curve(yt, pr[:, 1])
            aucv = roc_auc_score(yt, pr[:, 1])
            ax.plot(fpr, tpr, label=f"{model_name} (AUC={aucv:.3f})")
        else:
            yoh = np.eye(n_classes)[yt]
            grid = np.linspace(0, 1, 501)
            tprs, aucs = [], []
            for c in range(n_classes):
                fpr, tpr, _ = roc_curve(yoh[:, c], pr[:, c])
                tprs.append(np.interp(grid, fpr, tpr)); aucs.append(auc(fpr, tpr))
            ax.plot(grid, np.mean(tprs, axis=0), label=f"{model_name} (macro AUC={np.mean(aucs):.3f})")
    ax.plot([0, 1], [0, 1], "--", linewidth=1)
    ax.set_xlabel("False-positive rate"); ax.set_ylabel("True-positive rate")
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.set_title(f"ROC curves (seed {SEED}) \u2014 {dataset_name}"); ax.legend(fontsize=6)
    save_figure(fig, "roc_curves_all_models", dataset_name)


def draw_confusion_matrices(dataset_name, primary_result, class_names, normalize=False):
    n_classes = len(class_names)
    fig = plt.figure(figsize=(DOUBLE_COL_IN * 1.05, DOUBLE_COL_IN * 0.76))
    tag = "Row-normalized confusion matrices" if normalize else "Confusion matrices"
    fig.suptitle(f"{tag} (seed {SEED}) \u2014 {dataset_name}")
    gs = GridSpec(2, 3, figure=fig)
    for i, name in enumerate(MODEL_NAMES):
        ax = fig.add_subplot(gs[i // 3, i % 3])
        yt, yp, _ = primary_result["test_predictions"][name]
        cm = confusion_matrix(yt, yp, labels=np.arange(n_classes),
                               normalize="true" if normalize else None)
        ax.imshow(cm, vmin=0, vmax=1 if normalize else None, cmap="Blues")
        ax.set_title(name); ax.grid(False)
        ax.set_xticks(range(n_classes)); ax.set_yticks(range(n_classes))
        ax.set_xticklabels(class_names, rotation=45, ha="right"); ax.set_yticklabels(class_names)
        ax.set_xlabel("Predicted"); ax.set_ylabel("True")
        thr = cm.max() / 2 if cm.size else 0
        for r in range(cm.shape[0]):
            for c in range(cm.shape[1]):
                val = f"{cm[r, c]:.2f}" if normalize else str(int(cm[r, c]))
                ax.text(c, r, val, ha="center", va="center", fontsize=6.5,
                        color="white" if cm[r, c] > thr else "black")
    ax = fig.add_subplot(gs[1, 2]); ax.axis("off")
    ax.text(.02, .95, "Rows sum to 1.00" if normalize else "Counts on the held-out test split",
            va="top", fontsize=7)
    plt.tight_layout()
    save_figure(fig, "normalized_confusion_matrices" if normalize else "confusion_matrices", dataset_name)
    plt.show()
    

# =============================================================================
# XAI FIGURES (unchanged logic; use primary-seed best model)
# =============================================================================
def draw_training_curves(dataset_name, model_name, history):
    fig, ax = plt.subplots(figsize=(SINGLE_COL_IN * 1.3, SINGLE_COL_IN))
    e = np.arange(1, len(history["train_acc"]) + 1)
    ax.plot(e, history["train_acc"], marker="o", color=COLOR_MAX, label="Train acc")
    ax.plot(e, history["val_acc"], marker="s", color=COLOR_AVG, label="Val acc")
    ax.set_title(f"{model_name} \u2014 {dataset_name}"); ax.set_xlabel("Epoch"); ax.set_ylabel("Accuracy")
    ax.legend()
    save_figure(fig, f"training_curve_{model_name}", dataset_name)


def draw_figure2_pooling_behavior(dataset_name, bundle, best_model, best_model_name, xai_indices):
    """Figure 2 in the exact 3x4 panel order requested in the manuscript."""
    rep_local_idx = xai_indices[0]
    rep_x, _, rep_disp = bundle.get_sample(rep_local_idx)
    rep_x = rep_x.to(DEVICE)
    rep_rgb = rgb_from_display(rep_disp)
    with torch.no_grad():
        rep_pred = int(best_model(rep_x).argmax(1)[0])

    feat = capture_feature_map(best_model, target_layer_for(best_model_name, best_model), rep_x)
    fm = feat.mean(0, keepdim=True)[None]
    fm = F.interpolate(fm, (IMG_SIZE, IMG_SIZE), mode="bilinear", align_corners=False)[0, 0].numpy()
    fm = normalize_map(fm)

    # The two pooling views are derived from the same target-layer activation map.
    ft = torch.as_tensor(fm, dtype=torch.float32)[None, None]
    max_sal = normalize_map(F.max_pool2d(ft, 3, stride=1, padding=1)[0, 0].numpy())
    avg_sal = normalize_map(F.avg_pool2d(ft, 3, stride=1, padding=1)[0, 0].numpy())
    q = ALIGNMENT_SALIENCY_QUANTILE
    max_mask = max_sal >= np.quantile(max_sal, q)
    avg_mask = avg_sal >= np.quantile(avg_sal, q)
    overlap = max_mask & avg_mask
    max_only = max_mask & ~avg_mask
    avg_only = avg_mask & ~max_mask
    difference = normalize_map(np.abs(max_sal - avg_sal))

    # 3x3 sliding-window summary used for the bottom-left two panels.
    h, w = fm.shape
    re = np.linspace(0, h, 4, dtype=int); ce = np.linspace(0, w, 4, dtype=int)
    patch = np.zeros((3, 3), float)
    for r in range(3):
        for c in range(3):
            patch[r, c] = fm[re[r]:re[r+1], ce[c]:ce[c+1]].mean()
    patch = normalize_map(patch)
    local_max = patch == patch.max()
    local_avg = patch >= patch.mean()
    local_code = np.zeros_like(patch, dtype=int)  # 0 black, 1 max-only, 2 avg-only, 3 both
    local_code[local_max & ~local_avg] = 1
    local_code[local_avg & ~local_max] = 2
    local_code[local_max & local_avg] = 3
    local_rgb = np.zeros((*patch.shape, 3), float)
    local_rgb[local_code == 1] = (1, 0, 0)
    local_rgb[local_code == 2] = (0, 0, 1)
    local_rgb[local_code == 3] = (0, 1, 0)

    max_overlay = rep_rgb.copy(); max_overlay[max_mask] = 0.55 * max_overlay[max_mask] + 0.45 * np.array([1,0,0])
    max_overlay[~max_mask] = 0.70 * max_overlay[~max_mask] + 0.30 * np.array([0,0.65,0])
    avg_overlay = rep_rgb.copy(); avg_overlay[avg_mask] = 0.55 * avg_overlay[avg_mask] + 0.45 * np.array([1,0,0])
    avg_overlay[~avg_mask] = 0.70 * avg_overlay[~avg_mask] + 0.30 * np.array([0,0,1])
    combined = rep_rgb.copy()
    combined[max_only] = 0.55 * combined[max_only] + 0.45 * np.array([1,0,0])
    combined[avg_only] = 0.55 * combined[avg_only] + 0.45 * np.array([0,0,1])
    combined[overlap] = 0.50 * combined[overlap] + 0.50 * np.array([0.55,0.2,0.65])

    fig, axes = plt.subplots(3, 4, figsize=(10.5, 7.7))
    fig.suptitle(f"Figure 2: Layer-wise pooling behavior visualization -- {dataset_name}\n({best_model_name}, class = {bundle.class_names[rep_pred]})", fontsize=11, fontweight="bold", y=.985)

    def img(ax, a, title, **kw):
        ax.imshow(a, **kw); ax.set_title(title, fontsize=8, fontweight="bold"); ax.axis("off")

    img(axes[0,0], rep_rgb, "Original image")
    img(axes[0,1], fm, "Feature map +\nsliding window", cmap="viridis")
    axes[0,1].set_xticks([]); axes[0,1].set_yticks([])
    for z in (1/3, 2/3):
        axes[0,1].axhline(z*(IMG_SIZE-1), color="white", lw=.5, alpha=.7)
        axes[0,1].axvline(z*(IMG_SIZE-1), color="white", lw=.5, alpha=.7)
    axes[0,2].imshow(rep_rgb, cmap="gray", alpha=.55); axes[0,2].imshow(max_sal, cmap="hot", alpha=.65); axes[0,2].set_title("MaxPool saliency", fontsize=8, fontweight="bold"); axes[0,2].axis("off")
    axes[0,3].imshow(rep_rgb, cmap="gray", alpha=.60); axes[0,3].imshow(avg_sal, cmap="Reds", alpha=.45); axes[0,3].set_title("AvgPool saliency", fontsize=8, fontweight="bold"); axes[0,3].axis("off")

    img(axes[1,0], difference, "Difference\n(Max vs Avg)", cmap="cividis")
    img(axes[1,1], max_overlay, "Max: red=active,\ngreen=non-active")
    img(axes[1,2], avg_overlay, "Avg: red=active,\nblue=non-active")
    img(axes[1,3], combined, "Combined Max\nvs Avg focus")

    img(axes[2,0], patch, "Window zoom\n(step 24642)", cmap="viridis", interpolation="nearest")
    img(axes[2,1], local_rgb, "Local Max-only /\nAvg-only / Both", interpolation="nearest")
    vals = [100*max_mask.mean(), 100*avg_mask.mean(), 100*overlap.mean()]
    axes[2,2].bar(["Max\nactive", "Avg\nactive", "Overlap"], vals, color=COLOR_BAR)
    axes[2,2].set_ylim(0,100); axes[2,2].set_ylabel("% of feature map", fontsize=7); axes[2,2].set_title("Active pixel percentages", fontsize=8, fontweight="bold")
    for i,v in enumerate(vals): axes[2,2].text(i, v+2, f"{v:.1f}%", ha="center", fontsize=6)
    survive = [100*max_only.mean(), 100*(~max_mask).mean(), 100*avg_only.mean(), 100*(~avg_mask).mean()]
    axes[2,3].bar(["Max\nsurvive", "Max\nlost", "Avg\nsurvive", "Avg\nlost"], survive, color=COLOR_BAR)
    axes[2,3].set_ylim(0,100); axes[2,3].set_ylabel("%", fontsize=7); axes[2,3].set_title("Surviving vs lost features", fontsize=8, fontweight="bold")
    for i,v in enumerate(survive): axes[2,3].text(i, v+2, f"{v:.1f}%", ha="center", fontsize=6)

    plt.subplots_adjust(left=.04, right=.99, bottom=.06, top=.90, wspace=.42, hspace=.62)
    save_figure(fig, "figure2_pooling_behavior", dataset_name)
    plt.show()


def draw_figure5_sensitivity_probe(dataset_name, bundle, best_model, best_model_name, xai_indices):
    rep_local_idx = xai_indices[0]
    rep_x, _, _ = bundle.get_sample(rep_local_idx)
    rep_x = rep_x.to(DEVICE)
    methods, modes = ["DeepContour", "DeepEdge"], ["noise", "erase"]
    fig, axes = plt.subplots(2, 2, figsize=(DOUBLE_COL_IN * 1.1, DOUBLE_COL_IN * 0.9))
    fig.suptitle(f"Figure 5: Sensitivity probe (\u03b5={NOISE_STD:.2f}) \u2014 {dataset_name}", fontsize=10)
    for i, method in enumerate(methods):
        for j, mode in enumerate(modes):
            mx, av = sensitivity_curve(best_model, best_model_name, method, rep_x, mode)
            ax = axes[i, j]
            ax.plot(PERTURB_K, mx, marker="o", color=COLOR_MAX, label="Max")
            ax.plot(PERTURB_K, av, marker="o", color=COLOR_AVG, label="Avg")
            ax.set_title(f"{method} / {mode}", fontsize=8)
            ax.set_xlabel("Top-k% perturbed"); ax.set_ylabel("Mean |\u0394prob|")
            ax.set_xticks(PERTURB_K); ax.legend(fontsize=6)
    plt.tight_layout()
    save_figure(fig, "figure5_sensitivity_probe", dataset_name)
    plt.show()
    


# =============================================================================
# ABLATION: contour / edge / pooling / kernel-weight manipulation / fusion
# =============================================================================
def contour_map_from_rgb(rgb):
    g = gray(rgb).astype(np.float32)
    gy, gx = np.gradient(g)
    return normalize_map(np.abs(gx) + np.abs(gy))


def pooling_refinement(s):
    t = torch.as_tensor(normalize_map(s), dtype=torch.float32)[None, None]
    mx = F.max_pool2d(t, kernel_size=3, stride=1, padding=1)
    av = F.avg_pool2d(t, kernel_size=3, stride=1, padding=1)
    return normalize_map((0.5 * (mx + av))[0, 0].numpy())


@contextmanager
def temporary_kernel_scaling(layer, delta=KERNEL_WEIGHT_DELTA):
    if not hasattr(layer, "weight"):
        yield
        return
    with torch.no_grad():
        backup = layer.weight.detach().clone()
        layer.weight.mul_(1.0 + delta)
    try:
        yield
    finally:
        with torch.no_grad():
            layer.weight.copy_(backup)


def kernel_refinement(model, model_name, x, target, baseline_saliency):
    layer = target_layer_for(model_name, model)
    with temporary_kernel_scaling(layer):
        perturbed = cam_core(model, model_name, x, target, "GradCAM")
    return normalize_map(0.5 * normalize_map(baseline_saliency) + 0.5 * perturbed)


def weighted_fusion(S, C, E, use_parameterized=True, use_contour=True, use_edge=True):
    components, weights = [S], [FUSION_ALPHA if use_parameterized else 1.0]
    if use_contour:
        components.append(C); weights.append(FUSION_BETA if use_parameterized else 1.0)
    if use_edge:
        components.append(E); weights.append(FUSION_GAMMA if use_parameterized else 1.0)
    w = np.asarray(weights, float); w /= w.sum()
    fused = sum(wi * normalize_map(ci) for wi, ci in zip(w, components))
    return normalize_map(fused)


def make_ablation_map(model, model_name, x, xd, variant):
    with torch.no_grad():
        target = int(model(x).argmax(1)[0])
    S = cam_core(model, model_name, x, target, "GradCAM")
    rgb = rgb_from_display(xd)
    C, E = contour_map_from_rgb(rgb), edge_map(rgb)

    use_c = variant not in ("w/o contour", "Saliency only")
    use_e = variant not in ("w/o edge", "Saliency only")
    use_param = variant != "w/o parameterized fusion"
    use_pool = variant not in ("w/o pooling", "Saliency only")
    use_kernel = variant not in ("w/o kernel manipulation", "Saliency only")

    out = weighted_fusion(S, C, E, use_parameterized=use_param, use_contour=use_c, use_edge=use_e)
    if use_pool:
        pref = pooling_refinement(out)
        out = normalize_map((1 - POOL_REFINEMENT_STRENGTH) * out + POOL_REFINEMENT_STRENGTH * pref)
    if use_kernel:
        kref = kernel_refinement(model, model_name, x, target, S)
        out = normalize_map((1 - KERNEL_REFINEMENT_STRENGTH) * out + KERNEL_REFINEMENT_STRENGTH * kref)
    return out


def evaluate_ablation_all_seeds(dataset_name, bundle, per_seed):
    """Per-seed, per-sample-image ablation on that seed's best-accuracy model.
    Returns a raw DataFrame with a Seed column so every metric can be reported
    as Mean +/- SD [Min, Max] across REPEAT_SEEDS."""
    rows = []
    for seed, res in per_seed.items():
        best_model_name = pick_best_model_name(res)
        best_model = res["trained_models"][best_model_name]
        for local_idx in res["xai_indices"]:
            x, y, xd = bundle.get_sample(local_idx); x = x.to(DEVICE)
            ref = proxy_reference_mask(rgb_from_display(xd))
            for variant in ABLATION_VARIANTS:
                s = make_ablation_map(best_model, best_model_name, x, xd, variant)
                p, r, f1, iou = binary_metrics(binarize_saliency(s), ref)
                rows.append({
                    "Dataset": dataset_name, "Seed": seed, "Model": best_model_name,
                    "Sample": local_idx, "Variant": variant,
                    "Precision": p, "Recall": r, "F1": f1, "IoU": iou,
                    "DetectionStrength": detection_strength(s),
                    "CompositeTrust": float(np.mean([p, r, f1, iou])),
                })
    df = pd.DataFrame(rows)
    if SAVE_OUTPUTS:
        ABLATION_DIR.mkdir(parents=True, exist_ok=True)
        safe_ds = dataset_name.replace(" ", "_").replace("(", "").replace(")", "")
        df.to_csv(ABLATION_DIR / f"ablation_raw_{safe_ds}.csv", index=False)
    return df


def summarize_ablation_across_seeds(raw_df, metric="CompositeTrust"):
    """First average over sample images within a seed, then compute
    Mean/SD/Min/Max across the independent seeds."""
    per_seed_mean = raw_df.groupby(["Dataset", "Variant", "Seed"])[metric].mean().reset_index()
    rows = []
    for (ds, variant), g in per_seed_mean.groupby(["Dataset", "Variant"]):
        stat = mean_sd_min_max(g[metric].tolist())
        rows.append({"Dataset": ds, "Variant": variant, "Metric": metric, **stat})
    return pd.DataFrame(rows)


def draw_ablation_bar(summary, dataset_name, variants=None, seeds_label=""):
    sub = summary[summary.Dataset == dataset_name]
    order = [v for v in ABLATION_VARIANTS if variants is None or v in variants]
    sub = sub.set_index("Variant").loc[[v for v in order if v in sub.Variant.values]].reset_index()
    fig, ax = plt.subplots(figsize=(DOUBLE_COL_IN, SINGLE_COL_IN * .9))
    x = np.arange(len(sub))
    y = sub["Mean"].to_numpy(float)
    yerr = sub["SD"].to_numpy(float)
    ax.bar(x, y, yerr=yerr, capsize=4, color=COLOR_BAR)
    ax.set_xticks(x); ax.set_xticklabels(sub["Variant"], rotation=35, ha="right")
    ax.set_ylim(0, 1); ax.set_ylabel("Composite trust/alignment score")
    ax.set_title(f"Component ablation (Mean \u00b1 SD across seeds{seeds_label}) \u2014 {dataset_name}")
    save_figure(fig, "ablation_components", dataset_name)


def interactive_ablation_explorer(raw_df):
    """CLI explorer: pick a dataset, one or more ablation variants, and one or
    more REPEAT_SEEDS to immediately re-plot the comparison, without
    re-running any training or inference. Falls back gracefully (no crash)
    when stdin is not interactive (e.g. batch execution)."""
    datasets_available = sorted(raw_df["Dataset"].unique().tolist())
    if not datasets_available:
        return
    print("\n=== Interactive ablation explorer ===")
    print("Datasets:", ", ".join(datasets_available))
    print("Variants:", ", ".join(ABLATION_VARIANTS))
    print(f"Seeds:    {REPEAT_SEEDS}")
    print("Press Enter at any prompt to use ALL available options. Type 'quit' to exit.\n")
    while True:
        try:
            ds_in = input(f"Dataset [{datasets_available[0]}]: ").strip()
        except EOFError:
            print("(non-interactive session detected -- skipping explorer)")
            return
        if ds_in.lower() == "quit":
            return
        dataset_name = ds_in if ds_in in datasets_available else datasets_available[0]

        var_in = input("Variants (comma-separated, blank = all): ").strip()
        variants = [v.strip() for v in var_in.split(",") if v.strip()] if var_in else list(ABLATION_VARIANTS)
        variants = [v for v in variants if v in ABLATION_VARIANTS] or list(ABLATION_VARIANTS)

        seed_in = input("Seeds (comma-separated ints, blank = all): ").strip()
        try:
            seeds = [int(s) for s in seed_in.split(",") if s.strip()] if seed_in else list(REPEAT_SEEDS)
        except ValueError:
            seeds = list(REPEAT_SEEDS)
        seeds = [s for s in seeds if s in REPEAT_SEEDS] or list(REPEAT_SEEDS)

        subset = raw_df[(raw_df.Dataset == dataset_name) & (raw_df.Seed.isin(seeds))]
        if subset.empty:
            print("No matching rows -- try again.\n")
            continue
        summary = summarize_ablation_across_seeds(subset, metric="CompositeTrust")
        draw_ablation_bar(summary, dataset_name, variants=variants,
                           seeds_label=f" {seeds}")
        again = input("Explore another selection? [y/N]: ").strip().lower()
        if again != "y":
            return


def draw_alignment_error_panel(dataset_name, bundle, best_model, best_model_name, xai_indices):
    """Three-panel manuscript layout: Saliency Gray | DeepContour-like errors | metrics."""
    local_idx = xai_indices[0]
    x, _, xd = bundle.get_sample(local_idx); x = x.to(DEVICE)
    with torch.no_grad(): target = int(best_model(x).argmax(1)[0])
    sal = deep_contour_like(best_model, best_model_name, x, target)
    sm = binarize_saliency(sal).astype(bool)
    ref = proxy_reference_mask(rgb_from_display(xd)).astype(bool)
    tp, fp, fn = sm & ref, sm & ~ref, ~sm & ref
    err = np.zeros((*sm.shape, 3), dtype=float)
    err[tp] = (0.0, 1.0, 0.0)       # TP = green
    err[fp] = (1.0, 0.0, 0.0)       # FP = red
    err[fn] = (1.0, 0.0, 1.0)       # FN = magenta/blue-like
    p, r, f1, iou = binary_metrics(sm, ref)

    fig, axes = plt.subplots(1, 3, figsize=(10.5, 4.8), gridspec_kw={'width_ratios':[1.05,1.05,1.0]})
    axes[0].imshow(sal, cmap='magma'); axes[0].set_title('Saliency Gray', fontsize=14, fontweight='bold'); axes[0].axis('off')
    axes[1].imshow(err); axes[1].set_title('Errors (DeepContour-like)\nTP=G, FP=R, FN=B', fontsize=11, fontweight='bold'); axes[1].axis('off')
    vals=[p,r,f1,iou]
    axes[2].bar(['Precision','Recall','F1','IoU'], vals, color=COLOR_BAR)
    axes[2].set_ylim(0,1); axes[2].set_title('Alignment metrics', fontsize=14, fontweight='bold')
    axes[2].tick_params(axis='x', rotation=35)
    for i,v in enumerate(vals): axes[2].text(i, min(.98,v+.025), f'{v:.2f}', ha='center', fontsize=9)
    plt.tight_layout()
    save_figure(fig, 'alignment_errors_deepcontour_layout', dataset_name)


def _pooling_patch_codes(best_model, best_model_name, x):
    feat = capture_feature_map(best_model, target_layer_for(best_model_name, best_model), x)
    fm = normalize_map(feat.mean(0).numpy())
    h,w=fm.shape; re=np.linspace(0,h,4,dtype=int); ce=np.linspace(0,w,4,dtype=int)
    patch=np.array([[fm[re[r]:re[r+1],ce[c]:ce[c+1]].mean() for c in range(3)] for r in range(3)])
    patch=normalize_map(patch)
    # Max-survival: local maxima against 3x3 max pool; Avg-survival: >= local mean.
    t=torch.tensor(patch,dtype=torch.float32)[None,None]
    mx=F.max_pool2d(t,3,1,1)[0,0].numpy(); av=F.avg_pool2d(t,3,1,1)[0,0].numpy()
    max_surv=np.isclose(patch,mx,atol=1e-7)
    avg_surv=patch>=av
    return patch,max_surv,~max_surv,avg_surv,~avg_surv,(max_surv|avg_surv)


def draw_surviving_lost_features(dataset_name, bundle, best_model, best_model_name, xai_indices):
    """Exact 2x3 visual organization of the supplied lost/surviving-features figure."""
    x,_,_=bundle.get_sample(xai_indices[0]); x=x.to(DEVICE)
    patch,ms,ml,avs,avl,union=_pooling_patch_codes(best_model,best_model_name,x)
    fig,axes=plt.subplots(2,3,figsize=(10.7,6.5))
    panels=[(patch,'Window patch (normalized)','viridis'),(ms,'Max surviving features','gray_r'),(ml,'Max lost features','gray_r'),
            (avs,'Avg surviving features','gray_r'),(avl,'Avg lost features','gray_r'),(union,'Surviving (Max ∪ Avg)','gray_r')]
    for ax,(arr,title,cmap) in zip(axes.flat,panels):
        ax.imshow(arr,cmap=cmap,interpolation='nearest',vmin=0 if arr.dtype==bool else None,vmax=1 if arr.dtype==bool else None)
        ax.set_title(title,fontsize=11,fontweight='bold'); ax.axis('off')
    plt.tight_layout()
    save_figure(fig,'lost_and_surviving_features',dataset_name)


def kernel_weight_response(model, model_name, bundle, xai_indices):
    """Mean absolute target-probability change after +5% target-layer kernel scaling."""
    vals=[]
    layer=target_layer_for(model_name,model)
    for idx in xai_indices:
        x,_,_=bundle.get_sample(idx); x=x.to(DEVICE)
        with torch.no_grad():
            p0=F.softmax(model(x),1)[0]; target=int(p0.argmax())
        with temporary_kernel_scaling(layer, KERNEL_WEIGHT_DELTA):
            with torch.no_grad(): p1=F.softmax(model(x),1)[0]
        vals.append(abs(float(p1[target]-p0[target])))
    return float(np.mean(vals)) if vals else np.nan


def build_table8_kernel(dataset_name, bundle, per_seed):
    """Table 8: kernel-weight response for seeds 11,22,33,44,55; 6 decimals."""
    rows = []
    for model_name in MODEL_NAMES:
        row = {"Model": model_name}
        for seed in REPEAT_SEEDS:
            res = per_seed[seed]
            value = kernel_weight_response(
                res["trained_models"][model_name],
                model_name,
                bundle,
                res["xai_indices"],
            )
            row[f"Seed={seed}"] = f"{value:.6f}" if np.isfinite(value) else "n/a"
        rows.append(row)
    return pd.DataFrame(rows)


def binary_confusion_components(y_true,y_pred):
    cm=confusion_matrix(y_true,y_pred,labels=[0,1])
    if cm.shape!=(2,2): return np.nan,np.nan,np.nan,np.nan
    tn,fp,fn,tp=cm.ravel(); return int(tp),int(fn),int(fp),int(tn)


def build_table9_primary(dataset_name, primary_result):
    """
    Table 9: held-out confusion counts plus calibration/classification metrics.
    ECE, Brier, AUC and Accuracy are formatted to exactly six decimal places.
    Accuracy is reported as a percentage to preserve the uploaded-table format.
    """
    rows = []
    for m in MODEL_NAMES:
        yt, yp, pr = primary_result["test_predictions"][m]
        md = classification_metric_dict(yt, yp, pr)
        tp, fn, fp, tn = binary_confusion_components(yt, yp)
        rows.append({
            "Model": m,
            "TP": tp, "FN": fn, "FP": fp, "TN": tn,
            "ECE": f"{md['ECE']:.6f}" if np.isfinite(md["ECE"]) else "n/a",
            "Brier": f"{md['Brier']:.6f}" if np.isfinite(md["Brier"]) else "n/a",
            "AUC": f"{md['AUC']:.6f}" if np.isfinite(md["AUC"]) else "n/a",
            "Accuracy": f"{100.0 * md['Accuracy']:.6f}",
        })
    return pd.DataFrame(rows)


def pooling_ablation_scores(model, model_name, bundle, xai_indices):
    max_scores=[]; avg_scores=[]
    for idx in xai_indices:
        x,_,_=bundle.get_sample(idx); x=x.to(DEVICE)
        feat=capture_feature_map(model,target_layer_for(model_name,model),x)
        fm=torch.tensor(normalize_map(feat.mean(0).numpy()),dtype=torch.float32)[None,None]
        max_scores.append(float(F.max_pool2d(fm,3,1,1).mean()))
        avg_scores.append(float(F.avg_pool2d(fm,3,1,1).mean()))
    return float(np.mean(max_scores)),float(np.mean(avg_scores))


def build_table10_pooling(dataset_name, bundle, per_seed):
    """Table 10: MaxP/AvgP pooling ablation for every training seed."""
    rows = []
    for m in MODEL_NAMES:
        row = {"Model": m}
        for seed in REPEAT_SEEDS:
            mx, av = pooling_ablation_scores(
                per_seed[seed]["trained_models"][m],
                m, bundle, per_seed[seed]["xai_indices"]
            )
            row[f"Seed={seed} MaxP"] = f"{mx:.6f}" if np.isfinite(mx) else "n/a"
            row[f"Seed={seed} AvgP"] = f"{av:.6f}" if np.isfinite(av) else "n/a"
        rows.append(row)
    return pd.DataFrame(rows)


def render_manuscript_tables_8_9_10(dataset_name,bundle,per_seed,primary_result):
    t8=build_table8_kernel(dataset_name,bundle,per_seed)
    t9=build_table9_primary(dataset_name,primary_result)
    t10=build_table10_pooling(dataset_name,bundle,per_seed)
    _render_table_image(t8, f'Table 8: Kernel-weight manipulation across five seeds — {dataset_name}',
                        f'table8_kernel_weight_{dataset_name}', (DOUBLE_COL_IN*1.7,3.0),
                        note=f'Values are mean absolute target-probability changes; kernel scaling Δ={KERNEL_WEIGHT_DELTA:.2f}. Seeds={REPEAT_SEEDS}.')
    _render_table_image(t9, f'Table 9: Classification performance and calibration reliability — {dataset_name}',
                        f'table9_classification_calibration_{dataset_name}', (DOUBLE_COL_IN*1.65,3.0),
                        note=f'Held-out test split, primary seed={SEED}. Table 4 retains cross-seed Mean ± SD [Min, Max].')
    _render_table_image(t10, f'Table 10: Ablation analysis of pooling strategies — {dataset_name}',
                        f'table10_pooling_ablation_{dataset_name}', (DOUBLE_COL_IN*2.2,3.2),
                        note=f'MaxP/AvgP are activation scores computed independently for every seed {REPEAT_SEEDS}.')
    return t8,t9,t10

# =============================================================================
# MAIN
# =============================================================================
def main():
    bundles = load_available_datasets(DATASET_PATHS)
    all_ablation_raw = []

    for dataset_name, bundle in bundles.items():
        print(f"\n{'=' * 70}\nDATASET: {dataset_name}\n{'=' * 70}")

        run_cache = {}
        per_seed = run_all_seeds(bundle, MODEL_NAMES, cache=run_cache)
        primary_result = per_seed[SEED]
        best_model_name = pick_best_model_name(primary_result)
        best_model = primary_result["trained_models"][best_model_name]
        print(f"[{dataset_name}] Best-accuracy architecture (seed {SEED}): {best_model_name}")

        # ---- XAI qualitative figures (primary seed) ----
        for model_name in MODEL_NAMES:
            hist_model = model_name
            # histories are not retained post-hoc training loop return; recompute quickly is
            # unnecessary -- training curves are optional supplementary figures, so we skip
            # them here to keep the seed loop lean and focus compute on the requested tables.
        draw_figure2_pooling_behavior(dataset_name, bundle, best_model, best_model_name,
                                       primary_result["xai_indices"])
        draw_figure5_sensitivity_probe(dataset_name, bundle, best_model, best_model_name,
                                        primary_result["xai_indices"])
        draw_alignment_error_panel(dataset_name, bundle, best_model, best_model_name,
                                   primary_result["xai_indices"])
        draw_surviving_lost_features(dataset_name, bundle, best_model, best_model_name,
                                     primary_result["xai_indices"])
        table8, table9, table10 = render_manuscript_tables_8_9_10(
            dataset_name, bundle, per_seed, primary_result)

        # ---- Tables 1 and 2 (cross-seed Mean +/- SD [Min, Max]) ----
        tsm = build_technique_seed_metric(per_seed, best_model_name)
        table1 = build_table1(dataset_name, tsm)
        table2 = build_table2(dataset_name, tsm)
        _render_table_image(
            table1, f"Table 1: Alignment assessment \u2014 {dataset_name} ({best_model_name})",
            f"table1_alignment_{dataset_name}",
            figsize=(DOUBLE_COL_IN * 1.9, 1.4 + 0.35 * len(table1)),
            note=f"Mean \u00b1 SD [Min, Max] across independent seeds {REPEAT_SEEDS}.",
        )
        _render_table_image(
            table2, f"Table 2: Integrated evaluation \u2014 {dataset_name} ({best_model_name})",
            f"table2_integrated_{dataset_name}",
            figsize=(DOUBLE_COL_IN * 1.9, 1.4 + 0.35 * len(table2)),
            note=f"Mean \u00b1 SD [Min, Max] across independent seeds {REPEAT_SEEDS}.",
        )

        # ---- Table 3: uploaded localization/stability layout, cross-seed summaries ----
        table3 = render_table3(dataset_name, per_seed)

        # ---- Tables 11-12: corrected repeated-measures statistics and planned contrast ----
        table11, table12 = render_tables11_12(dataset_name, tsm)

        # ---- Table 4: classification metrics, calibration, confusion matrices ----
        cls_table = build_classification_table(dataset_name, per_seed)
        _render_table_image(
            cls_table, f"Table 4: Classification performance \u2014 {dataset_name}",
            f"table4_classification_{dataset_name}",
            figsize=(DOUBLE_COL_IN * 2.0, 1.4 + 0.35 * len(cls_table)),
            note=f"Mean \u00b1 SD [Min, Max] across seeds {REPEAT_SEEDS}.",
        )
        draw_reliability_diagrams(dataset_name, primary_result)
        draw_roc_curves(dataset_name, primary_result, bundle.num_classes)
        draw_confusion_matrices(dataset_name, primary_result, bundle.class_names, normalize=False)
        draw_confusion_matrices(dataset_name, primary_result, bundle.class_names, normalize=True)

        # ---- Ablation across seeds ----
        ablation_raw = evaluate_ablation_all_seeds(dataset_name, bundle, per_seed)
        all_ablation_raw.append(ablation_raw)
        for metric in ["Precision", "Recall", "F1", "IoU", "DetectionStrength", "CompositeTrust"]:
            summary = summarize_ablation_across_seeds(ablation_raw, metric=metric)
            if SAVE_OUTPUTS:
                ABLATION_DIR.mkdir(parents=True, exist_ok=True)
                safe_ds = dataset_name.replace(" ", "_").replace("(", "").replace(")", "")
                summary.to_csv(ABLATION_DIR / f"ablation_summary_{metric}_{safe_ds}.csv", index=False)
        composite_summary = summarize_ablation_across_seeds(ablation_raw, metric="CompositeTrust")
        draw_ablation_bar(composite_summary, dataset_name)

        print(f"\n[{dataset_name}] TABLE 1\n{table1.to_string(index=False)}")
        print(f"\n[{dataset_name}] TABLE 2\n{table2.to_string(index=False)}")
        print(f"\n[{dataset_name}] TABLE 4\n{cls_table.to_string(index=False)}")
        print(f"\n[{dataset_name}] TABLE 8\n{table8.to_string(index=False)}")
        print(f"\n[{dataset_name}] TABLE 9\n{table9.to_string(index=False)}")
        print(f"\n[{dataset_name}] TABLE 10\n{table10.to_string(index=False)}")
        print(f"\n[{dataset_name}] TABLE 11\n{table11.to_string(index=False)}")
        print(f"\n[{dataset_name}] TABLE 12\n{table12.to_string(index=False)}")

    if SAVE_OUTPUTS:
        run_meta = {
            "seed_primary": SEED, "repeat_seeds": REPEAT_SEEDS, "img_size": IMG_SIZE,
            "max_epochs": EPOCHS, "fixed_epoch_protocol": RUN_FIXED_30_EPOCHS, "early_stopping_patience": EARLY_STOPPING_PATIENCE, "early_stopping_min_delta": EARLY_STOPPING_MIN_DELTA, "torch_version": torch.__version__, "device": str(DEVICE),
            "perturbation_magnitude": PERTURBATION_MAGNITUDE,
            "perturbation_affected_pixel_percent": PERTURBATION_AFFECTED_PIXEL_PERCENT,
            "perturbation_repetitions": PERTURBATION_REPETITIONS,
            "perturbation_random_seeds": PERTURBATION_RANDOM_SEEDS,
            "statistical_dependent_variables": STATISTICAL_DVS,
            "statistical_independent_variable": "XAI technique (6 within-run levels)",
            "statistical_repeated_unit": "independent training seed",
            "multiplicity_procedure": MULTIPLICITY_PROCEDURE,
            "statistical_alpha": STAT_ALPHA,
            "effect_size": "partial eta-squared for omnibus ANOVA; Cohen dz for paired contrasts",
            "effect_size_ci": f"{STAT_BOOTSTRAP_REPS}-replicate seed-level bootstrap 95% CI for partial eta-squared; t-based 95% CI for paired mean difference",
            "perturbation_aggregation": PERTURBATION_AGGREGATION,
                        "summary_reporting": "No inferential significance testing. Tables 1, 2 and 4 and ablation summaries report descriptive Mean/SD/Min/Max across independent training seeds; Table 8 reports kernel-weight response for seeds 11,22,33,44,55 at six decimals; Table 9 reports primary-seed ECE/Brier/AUC/Accuracy at six decimals; Table 10 reports per-seed pooling diagnostics.",
        }
        TABLE_DIR.mkdir(parents=True, exist_ok=True)
        with open(TABLE_DIR / "run_metadata.json", "w") as f:
            json.dump(run_meta, f, indent=2)
        print(f"\nFigures  -> {FIG_DIR.resolve()}")
        print(f"Tables   -> {TABLE_DIR.resolve()}")
        print(f"Ablation -> {ABLATION_DIR.resolve()}")
        print(f"Predictions -> {PRED_DIR.resolve()}")
        print(f"Checkpoints -> {CKPT_DIR.resolve()}")

    plt.ioff()
    plt.show()

    if all_ablation_raw and SAVE_OUTPUTS:
        full_raw = pd.concat(all_ablation_raw, ignore_index=True)
        ABLATION_DIR.mkdir(parents=True, exist_ok=True)
        full_raw.to_csv(ABLATION_DIR / "ablation_all_datasets_raw.csv", index=False)

    print("\nDone. All figures/tables are derived from the trained models on this machine; "
          "nothing is hard-coded. Tables 1, 2 and 4 report Mean +/- SD "
          "(ddof=1) [Min, Max] across REPEAT_SEEDS =", REPEAT_SEEDS)


if __name__ == "__main__":
    main()

